# Module 2: RAG Pipeline

## Retrieval-Augmented Generation (RAG) with LangChain, Ollama, and ChromaDB

**Author:** Alwajeez Tech Training Team  
**Prerequisite:** Module 1 (LangChain Agents)  
**Environment:** Local machine with Ollama running at `localhost:11434`

---

### What is RAG?

**Retrieval-Augmented Generation (RAG)** is a technique that enhances Large Language Models (LLMs) by giving them access to external knowledge. Instead of relying solely on what the model learned during training, RAG retrieves relevant documents from a knowledge base and includes them in the prompt as context.

**Why RAG matters:**
- LLMs have a **knowledge cutoff** -- they do not know about events or data after their training date.
- LLMs can **hallucinate** -- they may generate plausible-sounding but incorrect information.
- RAG provides **grounded answers** -- the model answers based on actual documents you control.
- RAG is **cheaper than fine-tuning** -- you do not need to retrain the model to add new knowledge.

**The RAG Pipeline:**
```
User Question
    --> Retrieve relevant documents from vector store
    --> Combine documents + question into a prompt
    --> Send to LLM
    --> Return grounded answer
```

In this lab, we will build a complete RAG pipeline using:
- **LangChain** for orchestration
- **Ollama** (llama3.2:3b) for embeddings and LLM inference
- **ChromaDB** for vector storage and similarity search
- **Domain-specific documents** about Alwajeez Tech and ZATCA regulations

---

## Section 1: Setup & Verification

First, let us install the required packages and verify that our local services (ChromaDB, Ollama) are working correctly.

In [1]:
# Install required packages
# These are the core LangChain libraries plus ChromaDB for vector storage
# and additional loaders for PDF and web content.

!pip install \
    "langchain>=0.3.0" \
    "langchain-ollama>=0.3.0" \
    "langchain-chroma>=0.2.0" \
    "langchain-core>=0.3.0" \
    "langchain-text-splitters>=0.3.0" \
    "chromadb>=0.5.0" \
    "pypdf>=4.0.0" \
    "beautifulsoup4"

In [2]:
# -----------------------------------------------------------------
# Verify ChromaDB: create a test collection, add a document, query, delete
# -----------------------------------------------------------------
# ChromaDB is an open-source embedding database. It stores text as
# numerical vectors and lets us find similar text via vector similarity.

import chromadb

# Create an ephemeral (in-memory) client for testing
test_client = chromadb.Client()

# Create a test collection
test_collection = test_client.create_collection(name="test_collection")
print(f"Created collection: {test_collection.name}")

# Add a test document -- ChromaDB will auto-generate a simple embedding
test_collection.add(
    documents=["ZATCA stands for Zakat, Tax and Customs Authority."],
    metadatas=[{"source": "test"}],
    ids=["test_doc_1"]
)
print(f"Document count: {test_collection.count()}")

# Query the collection
results = test_collection.query(
    query_texts=["What is ZATCA?"],
    n_results=1
)
print(f"Query result: {results['documents'][0][0]}")

# Clean up
test_client.delete_collection(name="test_collection")
print("Test collection deleted. ChromaDB is working correctly!")

Created collection: test_collection


C:\Users\renad\.cache\chroma\onnx_models\all-MiniLM-L6-v2\onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:19<00:00, 4.34MiB/s]


Document count: 1
Query result: ZATCA stands for Zakat, Tax and Customs Authority.
Test collection deleted. ChromaDB is working correctly!


In [3]:
# -----------------------------------------------------------------
# Verify OllamaEmbeddings: create embeddings for a test sentence
# -----------------------------------------------------------------
# Embeddings convert text into numerical vectors (lists of floats).
# Similar text will have similar vectors, which enables semantic search.
#
# IMPORTANT: Make sure Ollama is running locally on port 11434
# and the llama3.2:3b model is pulled:
#   ollama pull llama3.2:3b

from langchain_ollama import OllamaEmbeddings

# Create the embeddings model -- this connects to your local Ollama instance
embeddings = OllamaEmbeddings(
    model="llama3.2:3b",
    base_url="http://localhost:11434"
)

# Generate an embedding for a test sentence
test_text = "Alwajeez Tech builds AI-powered business solutions in Saudi Arabia."
test_vector = embeddings.embed_query(test_text)

print(f"Input text: '{test_text}'")
print(f"Vector length (dimensions): {len(test_vector)}")
print(f"First 10 values: {test_vector[:10]}")
print(f"\nOllamaEmbeddings is working correctly!")

d:\AI\agentic-ai-system\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Input text: 'Alwajeez Tech builds AI-powered business solutions in Saudi Arabia.'
Vector length (dimensions): 3072
First 10 values: [-0.025113057, -0.0059083705, -0.017687595, 0.0017470563, 0.010480608, -0.018701952, 0.003299512, 0.025228435, -0.0087515, 0.0038702744]

OllamaEmbeddings is working correctly!


---

## Section 2: Create Sample Documents

RAG requires a **knowledge base** -- a collection of documents that the model can reference when answering questions. In a real system, these might be company wikis, product manuals, regulatory documents, or customer support transcripts.

For this lab, we will create two domain-specific documents about:
1. **Alwajeez Tech Company Overview** -- company info, products, architecture, tools
2. **FAQ Document** -- frequently asked questions about the system, ZATCA, VAT, and deployment

We write these programmatically so the lab is fully reproducible.

In [4]:
# -----------------------------------------------------------------
# Programmatically create the sample documents
# -----------------------------------------------------------------
import os

# Create the sample_documents directory (relative to this notebook)
docs_dir = os.path.join(os.getcwd(), "sample_documents")
os.makedirs(docs_dir, exist_ok=True)

# --- Document 1: Company Overview ---
company_overview = """Alwajeez Tech - Company Overview
=================================

About Alwajeez Tech
-------------------
Alwajeez Tech is a technology company based in Saudi Arabia specializing in AI-powered business solutions. The company focuses on developing intelligent systems that help businesses automate their operations, comply with government regulations, and improve efficiency.

Core Products and Services
--------------------------
1. Agentic AI System: An intelligent agent platform built with FastAPI and Ollama that provides autonomous task execution, tool calling, and workflow automation. The system uses a ReAct (Reasoning + Acting) agent loop to break down complex tasks into steps.

2. ZATCA Compliance Solutions: Automated e-invoicing and tax compliance tools that help businesses comply with Saudi Arabia's Zakat, Tax and Customs Authority (ZATCA) regulations. This includes Phase 2 integration for real-time invoice reporting.

3. Workflow Automation: Integration with n8n for building automated business workflows that connect the AI agent with external services, databases, and APIs.

Technical Architecture
----------------------
The Agentic AI System consists of the following components:

- FastAPI Server (Port 8000): The main API server that exposes OpenAI-compatible endpoints for chat completions, agent execution, and tool management. It implements a custom ReAct agent with support for multiple built-in tools.

- Ollama LLM Backend (Port 11434): Runs large language models locally, providing fast inference without cloud API dependencies. The default model is llama3.2:3b, optimized for instruction following and tool use.

- n8n Workflow Engine (Port 5678): A visual workflow automation platform that connects to the AI agent via webhooks. It enables building complex business processes that combine AI reasoning with external service integrations.

- Redis Cache (Port 6379): Provides caching for frequently used data and serves as a message queue for asynchronous task processing.

Built-in Tools
--------------
The Agentic AI System includes the following built-in tools:

- Calculator: Evaluates mathematical expressions safely.
- Web Search: Searches the internet for current information (requires API integration).
- File Operations: Read, write, and list files within designated directories.
- HTTP Request: Makes HTTP requests to external APIs.
- ZATCA Compliance: Validates and generates ZATCA-compliant e-invoices.
- Webhook Sender: Sends data to n8n webhooks for workflow automation.
- Time Service: Gets the current date and time in any timezone.

Team and Location
-----------------
Alwajeez Tech is headquartered in Riyadh, Saudi Arabia. The development team includes AI engineers, full-stack developers, and domain experts in Saudi regulatory compliance. The company serves clients across the GCC region.

Contact Information
-------------------
Website: www.alwajeez.tech
Location: Riyadh, Saudi Arabia
Timezone: Asia/Riyadh (UTC+3)
"""

# --- Document 2: FAQ ---
faq_content = """Alwajeez Tech - Frequently Asked Questions (FAQ)
==================================================

Q1: What is the Agentic AI System?
A1: The Agentic AI System is an intelligent agent platform developed by Alwajeez Tech. It uses a ReAct (Reasoning + Acting) loop to autonomously break down tasks, call tools, and produce results. It is built with FastAPI for the API layer and uses Ollama for local LLM inference. The system is fully self-hosted and does not require external cloud AI APIs.

Q2: What models does the system support?
A2: The system primarily uses Ollama for local model inference. The default model is llama3.2:3b. You can use any model available through Ollama including llama3.2:1b, llama3.1:8b, mistral:7b, codellama, and custom fine-tuned models. Models are pulled automatically when first requested.

Q3: How does the ReAct agent work?
A3: The ReAct agent follows a loop of Thought -> Action -> Observation. First, it thinks about the task and decides what to do. Then it takes an action by calling a tool (calculator, web search, file operations, etc.). It observes the result, then decides whether to take another action or provide a final answer. The maximum number of iterations is configurable (default: 10).

Q4: What is ZATCA and why is it important?
A4: ZATCA stands for Zakat, Tax and Customs Authority, the Saudi Arabian government body responsible for tax collection and customs regulation. ZATCA mandates electronic invoicing (e-invoicing) for all businesses in Saudi Arabia. Phase 1 requires generating invoices in a specific format, while Phase 2 requires real-time integration with ZATCA's systems for invoice clearance and reporting.

Q5: What is the VAT rate in Saudi Arabia?
A5: The standard VAT (Value Added Tax) rate in Saudi Arabia is 15%. This rate was increased from 5% to 15% in July 2020. Certain supplies are zero-rated (0%) such as exports and international transportation, while some supplies are exempt from VAT entirely, such as financial services and residential real estate.

Q6: How do I connect to the API?
A6: The FastAPI server runs on port 8000 and provides OpenAI-compatible endpoints. You can send chat requests to POST /v1/chat/completions, run agent tasks via POST /v1/agent/run, list available tools at GET /v1/tools, and check system health at GET /health. All endpoints accept and return JSON.

Q7: How does the tool system work?
A7: Tools are defined as ToolDefinition objects with a name, description, and parameter schema. The ToolExecutor class handles execution of each tool. When the agent decides to use a tool, it formats an ACTION line with the tool name and parameters. The system parses this, executes the tool, and returns the result as an OBSERVATION to the agent.

Q8: What is n8n and how does it integrate?
A8: n8n is an open-source workflow automation platform that runs on port 5678. It connects to the Agentic AI System via webhooks. When an event occurs in n8n, it can trigger the AI agent to process a task. Conversely, the agent can use the send_webhook tool to push data into n8n workflows. This enables building complex automation pipelines combining AI reasoning with external services.

Q9: What are the system requirements?
A9: The system requires Docker and Docker Compose for deployment. For GPU acceleration with Ollama, an NVIDIA GPU with CUDA support is recommended. Minimum requirements: 8GB RAM, 20GB disk space. For optimal performance with llama3.2:3b: 16GB RAM, NVIDIA GPU with 4GB+ VRAM.

Q10: How is the system deployed?
A10: The system is deployed using Docker Compose. The docker-compose.yml file defines four services: Ollama (LLM backend), Agentic API (FastAPI server), n8n (workflow automation), and Redis (caching). A model-loader service automatically pulls the required LLM model on first startup. All services communicate over a shared Docker network.

Q11: Can I fine-tune models for specific domains?
A11: Yes, you can fine-tune models using techniques like LoRA (Low-Rank Adaptation) and QLoRA (Quantized LoRA). The fine-tuned model can be exported in GGUF format and imported into Ollama using a Modelfile. This allows you to create domain-specific models for ZATCA compliance, customer support, or other specialized tasks while keeping inference local.

Q12: What is the difference between the chat endpoint and the agent endpoint?
A12: The chat endpoint (/v1/chat/completions) provides direct LLM inference - it sends your message to the model and returns the response. The agent endpoint (/v1/agent/run) uses the ReAct loop, which means the AI can reason about the task, call tools, observe results, and iterate until it reaches a final answer. Use chat for simple questions and agent for complex tasks requiring tool use.
"""

# Write the files
company_path = os.path.join(docs_dir, "company_overview.txt")
faq_path = os.path.join(docs_dir, "faq.txt")

with open(company_path, "w", encoding="utf-8") as f:
    f.write(company_overview)

with open(faq_path, "w", encoding="utf-8") as f:
    f.write(faq_content)

print(f"Created: {company_path}")
print(f"Created: {faq_path}")
print(f"\nFiles in {docs_dir}:")
for fname in os.listdir(docs_dir):
    fpath = os.path.join(docs_dir, fname)
    size = os.path.getsize(fpath)
    print(f"  - {fname} ({size:,} bytes)")

Created: d:\AI\agentic-ai-system\training\module2_rag\sample_documents\company_overview.txt
Created: d:\AI\agentic-ai-system\training\module2_rag\sample_documents\faq.txt

Files in d:\AI\agentic-ai-system\training\module2_rag\sample_documents:
  - company_overview.txt (3,021 bytes)
  - faq.txt (4,782 bytes)


In [5]:
# -----------------------------------------------------------------
# Verify files exist and preview their content
# -----------------------------------------------------------------
import os

docs_dir = os.path.join(os.getcwd(), "sample_documents")

for filename in ["company_overview.txt", "faq.txt"]:
    filepath = os.path.join(docs_dir, filename)
    assert os.path.exists(filepath), f"File not found: {filepath}"
    
    with open(filepath, "r", encoding="utf-8") as f:
        content = f.read()
    
    print(f"=== {filename} ===")
    print(f"Total length: {len(content)} characters")
    print(f"First 200 chars:\n{content[:200]}")
    print()

=== company_overview.txt ===
Total length: 2973 characters
First 200 chars:
Alwajeez Tech - Company Overview

About Alwajeez Tech
-------------------
Alwajeez Tech is a technology company based in Saudi Arabia specializing in AI-powered busin

=== faq.txt ===
Total length: 4744 characters
First 200 chars:
Alwajeez Tech - Frequently Asked Questions (FAQ)

Q1: What is the Agentic AI System?
A1: The Agentic AI System is an intelligent agent platform devel



---

## Section 3: Document Loading

LangChain provides **document loaders** that read data from various sources (text files, PDFs, web pages, databases) and convert them into `Document` objects. Each `Document` has:
- `page_content` (str) -- the text content
- `metadata` (dict) -- information about the source (file path, page number, etc.)

Common loaders:
| Loader | Source | Package |
|--------|--------|---------|
| `TextLoader` | `.txt` files | `langchain_community` |
| `DirectoryLoader` | All files in a folder | `langchain_community` |
| `PyPDFLoader` | `.pdf` files | `langchain_community` (requires `pypdf`) |
| `WebBaseLoader` | Web pages | `langchain_community` (requires `beautifulsoup4`) |

In [6]:
# -----------------------------------------------------------------
# TextLoader: Load a single text file
# -----------------------------------------------------------------
import os
from langchain_community.document_loaders import TextLoader

docs_dir = os.path.join(os.getcwd(), "sample_documents")

# Load the company overview file
loader = TextLoader(
    file_path=os.path.join(docs_dir, "company_overview.txt"),
    encoding="utf-8"
)
documents = loader.load()

print(f"Number of documents loaded: {len(documents)}")
print(f"Document type: {type(documents[0]).__name__}")
print(f"\nMetadata: {documents[0].metadata}")
print(f"\nContent length: {len(documents[0].page_content)} characters")
print(f"\nFirst 300 characters:")
print(documents[0].page_content[:300])

Number of documents loaded: 1
Document type: Document

Metadata: {'source': 'd:\\AI\\agentic-ai-system\\training\\module2_rag\\sample_documents\\company_overview.txt'}

Content length: 2973 characters

First 300 characters:
Alwajeez Tech - Company Overview

About Alwajeez Tech
-------------------
Alwajeez Tech is a technology company based in Saudi Arabia specializing in AI-powered business solutions. The company focuses on developing intelligent systems that help businesses automate t


In [7]:
# -----------------------------------------------------------------
# DirectoryLoader: Load all .txt files from a directory
# -----------------------------------------------------------------
import os
from langchain_community.document_loaders import DirectoryLoader, TextLoader

docs_dir = os.path.join(os.getcwd(), "sample_documents")

# Load all .txt files from the sample_documents directory
dir_loader = DirectoryLoader(
    path=docs_dir,
    glob="**/*.txt",           # Pattern to match text files
    loader_cls=TextLoader,      # Use TextLoader for each file
    loader_kwargs={"encoding": "utf-8"},
    show_progress=True
)
all_documents = dir_loader.load()

print(f"Total documents loaded: {len(all_documents)}")
print()
for i, doc in enumerate(all_documents):
    source = doc.metadata.get("source", "unknown")
    print(f"Document {i + 1}:")
    print(f"  Source: {source}")
    print(f"  Content length: {len(doc.page_content)} characters")
    print(f"  Preview: {doc.page_content[:100]}...")
    print()

100%|██████████| 2/2 [00:00<00:00, 363.62it/s]

Total documents loaded: 2

Document 1:
  Source: d:\AI\agentic-ai-system\training\module2_rag\sample_documents\company_overview.txt
  Content length: 2973 characters
  Preview: Alwajeez Tech - Company Overview

About Alwajeez Tech
------------...

Document 2:
  Source: d:\AI\agentic-ai-system\training\module2_rag\sample_documents\faq.txt
  Content length: 4744 characters
  Preview: Alwajeez Tech - Frequently Asked Questions (FAQ)
...



In [ ]:
# -----------------------------------------------------------------
# (Optional) PyPDFLoader: Load PDF files
# -----------------------------------------------------------------
# PyPDFLoader splits each page of a PDF into a separate Document.
# This is useful for large documents where you want per-page retrieval.
#
# Uncomment the code below if you have a PDF file to test with.

try:
    from langchain_community.document_loaders import PyPDFLoader
    
    # Example usage (uncomment and provide a real PDF path):
    # pdf_loader = PyPDFLoader("path/to/your/document.pdf")
    # pdf_docs = pdf_loader.load()
    # print(f"PDF pages loaded: {len(pdf_docs)}")
    # for doc in pdf_docs:
    #     print(f"  Page {doc.metadata['page']}: {len(doc.page_content)} chars")
    
    print("PyPDFLoader is available. Provide a PDF path to test it.")
    
except ImportError:
    print("PyPDFLoader not available. Install pypdf: pip install pypdf")

In [ ]:
# -----------------------------------------------------------------
# (Optional) WebBaseLoader: Load content from web pages
# -----------------------------------------------------------------
# WebBaseLoader fetches a web page and extracts text using BeautifulSoup.
# Useful for ingesting documentation sites, knowledge bases, etc.
#
# Note: Requires internet access and beautifulsoup4.

try:
    from langchain_community.document_loaders import WebBaseLoader
    
    # Example usage (uncomment to test with a real URL):
    # web_loader = WebBaseLoader("https://python.langchain.com/docs/concepts/rag/")
    # web_docs = web_loader.load()
    # print(f"Web documents loaded: {len(web_docs)}")
    # print(f"Content length: {len(web_docs[0].page_content)} characters")
    # print(f"First 300 chars: {web_docs[0].page_content[:300]}")
    
    print("WebBaseLoader is available. Provide a URL to test it.")
    
except ImportError:
    print("WebBaseLoader not available. Install beautifulsoup4: pip install beautifulsoup4")

---

## Section 4: Chunking Strategies

### Why Chunking Matters

Documents are often too long to fit into an LLM's context window or to be meaningfully compared via embeddings. **Chunking** splits documents into smaller pieces that can be individually embedded and retrieved.

**Key trade-offs:**
- **Small chunks** (100-300 chars): More precise retrieval, but may lose context. A chunk might contain only part of a sentence.
- **Medium chunks** (300-800 chars): Good balance of precision and context. Usually contains 1-3 complete paragraphs.
- **Large chunks** (800-2000 chars): More context per chunk, but retrieval may be less precise. May include irrelevant information.

**Chunk overlap** ensures that information at chunk boundaries is not lost. If a key fact spans two chunks, the overlap ensures both chunks contain it.

### RecursiveCharacterTextSplitter

LangChain's `RecursiveCharacterTextSplitter` is the recommended default. It tries to split on natural boundaries in this order:
1. `\n\n` (paragraph breaks)
2. `\n` (line breaks)
3. ` ` (spaces)
4. `` (individual characters)

This preserves semantic structure better than splitting at arbitrary character positions.

In [8]:
# -----------------------------------------------------------------
# Load documents and split with 4 different chunking configurations
# -----------------------------------------------------------------
import os
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Load all documents
docs_dir = os.path.join(os.getcwd(), "sample_documents")
dir_loader = DirectoryLoader(
    path=docs_dir,
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"}
)
raw_documents = dir_loader.load()
print(f"Loaded {len(raw_documents)} raw documents")
print(f"Total characters: {sum(len(d.page_content) for d in raw_documents):,}")

# Define 4 chunking configurations
chunking_configs = {
    "Config A (100/20)": {"chunk_size": 100, "chunk_overlap": 20},
    "Config B (300/50)": {"chunk_size": 300, "chunk_overlap": 50},
    "Config C (500/100)": {"chunk_size": 500, "chunk_overlap": 100},
    "Config D (1000/200)": {"chunk_size": 1000, "chunk_overlap": 200},
}

# Split documents with each configuration and store results
chunking_results = {}

for config_name, params in chunking_configs.items():
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=params["chunk_size"],
        chunk_overlap=params["chunk_overlap"],
        length_function=len,
        add_start_index=True,  # Track where each chunk starts in the original doc
    )
    chunks = splitter.split_documents(raw_documents)
    chunking_results[config_name] = chunks
    print(f"\n{config_name}: {len(chunks)} chunks")

Loaded 2 raw documents
Total characters: 7,717

Config A (100/20): 110 chunks

Config B (300/50): 49 chunks

Config C (500/100): 20 chunks

Config D (1000/200): 10 chunks


In [9]:
# -----------------------------------------------------------------
# Analyze each chunking configuration
# -----------------------------------------------------------------
for config_name, chunks in chunking_results.items():
    sizes = [len(chunk.page_content) for chunk in chunks]
    avg_size = sum(sizes) / len(sizes) if sizes else 0
    
    print(f"\n{'=' * 60}")
    print(f"{config_name}")
    print(f"{'=' * 60}")
    print(f"  Number of chunks: {len(chunks)}")
    print(f"  Average chunk size: {avg_size:.0f} characters")
    print(f"  Min chunk size: {min(sizes)} characters")
    print(f"  Max chunk size: {max(sizes)} characters")
    print(f"\n  --- Sample chunk (first chunk) ---")
    print(f"  {repr(chunks[0].page_content[:150])}")
    print(f"  Metadata: {chunks[0].metadata}")


Config A (100/20)
  Number of chunks: 110
  Average chunk size: 78 characters
  Min chunk size: 19 characters
  Max chunk size: 99 characters

  --- Sample chunk (first chunk) ---
  'Alwajeez Tech - Company Overview\n================================='
  Metadata: {'source': 'd:\\AI\\agentic-ai-system\\training\\module2_rag\\sample_documents\\company_overview.txt', 'start_index': 0}

Config B (300/50)
  Number of chunks: 49
  Average chunk size: 166 characters
  Min chunk size: 32 characters
  Max chunk size: 299 characters

  --- Sample chunk (first chunk) ---
  'Alwajeez Tech - Company Overview\n================================='
  Metadata: {'source': 'd:\\AI\\agentic-ai-system\\training\\module2_rag\\sample_documents\\company_overview.txt', 'start_index': 0}

Config C (500/100)
  Number of chunks: 20
  Average chunk size: 384 characters
  Min chunk size: 224 characters
  Max chunk size: 492 characters

  --- Sample chunk (first chunk) ---
  'Alwajeez Tech - Company Overview\n======

In [10]:
# -----------------------------------------------------------------
# Comparison summary table
# -----------------------------------------------------------------
print(f"{'Config':<25} {'Chunks':>8} {'Avg Size':>10} {'Min':>6} {'Max':>6}")
print("-" * 60)

for config_name, chunks in chunking_results.items():
    sizes = [len(chunk.page_content) for chunk in chunks]
    avg_size = sum(sizes) / len(sizes) if sizes else 0
    print(
        f"{config_name:<25} {len(chunks):>8} {avg_size:>10.0f} "
        f"{min(sizes):>6} {max(sizes):>6}"
    )

print()
print("Observations:")
print("- Smaller chunk sizes produce more chunks with finer granularity.")
print("- Larger chunk sizes preserve more context per chunk but produce fewer chunks.")
print("- Config C (500/100) is a common default that balances precision and context.")
print("- The best config depends on your use case -- experiment and evaluate!")

Config                      Chunks   Avg Size    Min    Max
------------------------------------------------------------
Config A (100/20)              110         78     19     99
Config B (300/50)               49        166     32    299
Config C (500/100)              20        384    224    492
Config D (1000/200)             10        770    127    969

Observations:
- Smaller chunk sizes produce more chunks with finer granularity.
- Larger chunk sizes preserve more context per chunk but produce fewer chunks.
- Config C (500/100) is a common default that balances precision and context.
- The best config depends on your use case -- experiment and evaluate!


---

## Section 5: Embeddings + ChromaDB Vector Store

### What Are Embeddings?

**Embeddings** are numerical representations (vectors) of text. They capture the *meaning* of text in a way that allows mathematical comparison:
- Similar text produces vectors that are close together (high cosine similarity).
- Dissimilar text produces vectors that are far apart (low cosine similarity).

### Vector Stores

A **vector store** (like ChromaDB) indexes these embeddings so you can efficiently find the most similar documents to a given query. The workflow is:

1. **Indexing:** Convert each chunk into an embedding vector and store it.
2. **Querying:** Convert the user's question into an embedding and find the closest stored vectors.
3. **Retrieval:** Return the original text chunks associated with the closest vectors.

```
"What is ZATCA?" --> [0.12, -0.34, 0.56, ...] --> Find closest vectors --> Return matching chunks
```

In [11]:
# -----------------------------------------------------------------
# Create OllamaEmbeddings instance
# -----------------------------------------------------------------
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(
    model="llama3.2:3b",
    base_url="http://localhost:11434"
)

# Quick test: embed two sentences and compare
vec1 = embeddings.embed_query("ZATCA is the Saudi tax authority.")
vec2 = embeddings.embed_query("The VAT rate in Saudi Arabia is 15%.")
vec3 = embeddings.embed_query("Python is a programming language.")

# Compute cosine similarity manually to demonstrate the concept
import math

def cosine_similarity(a, b):
    dot_product = sum(x * y for x, y in zip(a, b))
    magnitude_a = math.sqrt(sum(x * x for x in a))
    magnitude_b = math.sqrt(sum(x * x for x in b))
    if magnitude_a == 0 or magnitude_b == 0:
        return 0.0
    return dot_product / (magnitude_a * magnitude_b)

sim_12 = cosine_similarity(vec1, vec2)
sim_13 = cosine_similarity(vec1, vec3)

print(f"Similarity (ZATCA <-> VAT rate):  {sim_12:.4f}  -- related topics")
print(f"Similarity (ZATCA <-> Python):    {sim_13:.4f}  -- unrelated topics")
print(f"\nAs expected, related topics have higher similarity scores.")

Similarity (ZATCA <-> VAT rate):  0.7358  -- related topics
Similarity (ZATCA <-> Python):    0.6581  -- unrelated topics

As expected, related topics have higher similarity scores.


In [12]:
# -----------------------------------------------------------------
# Create ChromaDB vector store from documents
# -----------------------------------------------------------------
# We use Config C (500/100) as a good default chunking strategy.

import os
import shutil
from langchain_chroma import Chroma

# Use Config C chunks from Section 4
chunks_for_vectorstore = chunking_results["Config C (500/100)"]
print(f"Indexing {len(chunks_for_vectorstore)} chunks into ChromaDB...")

# Clean up any previous database
persist_directory = os.path.join(os.getcwd(), "chroma_db")
if os.path.exists(persist_directory):
    shutil.rmtree(persist_directory)

# Create the vector store -- this embeds all chunks and stores them
vectorstore = Chroma.from_documents(
    documents=chunks_for_vectorstore,
    embedding=embeddings,
    persist_directory=persist_directory,
    collection_name="alwajeez_docs"
)

print(f"Vector store created at: {persist_directory}")
print(f"Collection size: {vectorstore._collection.count()} documents")

Indexing 20 chunks into ChromaDB...
Vector store created at: d:\AI\agentic-ai-system\training\module2_rag\chroma_db
Collection size: 20 documents


In [13]:
# -----------------------------------------------------------------
# Test similarity_search: find the top 3 most relevant chunks
# -----------------------------------------------------------------
query = "What is ZATCA?"
print(f"Query: '{query}'")
print(f"{'=' * 60}")

results = vectorstore.similarity_search(query, k=3)

for i, doc in enumerate(results):
    print(f"\n--- Result {i + 1} ---")
    print(f"Source: {doc.metadata.get('source', 'unknown')}")
    print(f"Start index: {doc.metadata.get('start_index', 'N/A')}")
    print(f"Content ({len(doc.page_content)} chars):")
    print(doc.page_content)

Query: 'What is ZATCA?'

--- Result 1 ---
Source: d:\AI\agentic-ai-system\training\module2_rag\sample_documents\faq.txt
Start index: 1238
Content (434 chars):
Q4: What is ZATCA and why is it important?
A4: ZATCA stands for Zakat, Tax and Customs Authority, the Saudi Arabian government body responsible for tax collection and customs regulation. ZATCA mandates electronic invoicing (e-invoicing) for all businesses in Saudi Arabia. Phase 1 requires generating invoices in a specific format, while Phase 2 requires real-time integration with ZATCA's systems for invoice clearance and reporting.

--- Result 2 ---
Source: d:\AI\agentic-ai-system\training\module2_rag\sample_documents\faq.txt
Start index: 0
Content (492 chars):
Alwajeez Tech - Frequently Asked Questions (FAQ)

Q1: What is the Agentic AI System?
A1: The Agentic AI System is an intelligent agent platform developed by Alwajeez Tech. It uses a ReAct (Reasoning + Acting) loop to autonomously break down tasks, call tools, and produce re

In [14]:
# -----------------------------------------------------------------
# Test similarity_search_with_score: includes distance scores
# -----------------------------------------------------------------
# Lower distance = more similar (ChromaDB uses L2 distance by default)

query = "What is the VAT rate in Saudi Arabia?"
print(f"Query: '{query}'")
print(f"{'=' * 60}")

results_with_scores = vectorstore.similarity_search_with_score(query, k=3)

for i, (doc, score) in enumerate(results_with_scores):
    print(f"\n--- Result {i + 1} (distance: {score:.4f}) ---")
    print(f"Source: {doc.metadata.get('source', 'unknown')}")
    print(f"Content preview: {doc.page_content[:200]}...")

print(f"\nNote: Lower distance scores indicate higher relevance.")

Query: 'What is the VAT rate in Saudi Arabia?'

--- Result 1 (distance: 1.0496) ---
Source: d:\AI\agentic-ai-system\training\module2_rag\sample_documents\faq.txt
Content preview: Q5: What is the VAT rate in Saudi Arabia?
A5: The standard VAT (Value Added Tax) rate in Saudi Arabia is 15%. This rate was increased from 5% to 15% in July 2020. Certain supplies are zero-rated (0%) ...

--- Result 2 (distance: 1.3241) ---
Source: d:\AI\agentic-ai-system\training\module2_rag\sample_documents\faq.txt
Content preview: Q4: What is ZATCA and why is it important?
A4: ZATCA stands for Zakat, Tax and Customs Authority, the Saudi Arabian government body responsible for tax collection and customs regulation. ZATCA mandate...

--- Result 3 (distance: 1.4481) ---
Source: d:\AI\agentic-ai-system\training\module2_rag\sample_documents\company_overview.txt
Content preview: - Calculator: Evaluates mathematical expressions safely.
- Web Search: Searches the internet for current information (requires API integ

In [15]:
# -----------------------------------------------------------------
# Persist and reload from disk
# -----------------------------------------------------------------
# ChromaDB with persist_directory automatically saves to disk.
# Here we demonstrate loading an existing vector store from disk.

import os
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings

persist_directory = os.path.join(os.getcwd(), "chroma_db")

# Create a new embeddings instance (simulating a fresh session)
embeddings_reload = OllamaEmbeddings(
    model="llama3.2:3b",
    base_url="http://localhost:11434"
)

# Reload the vector store from disk
reloaded_vectorstore = Chroma(
    persist_directory=persist_directory,
    embedding_function=embeddings_reload,
    collection_name="alwajeez_docs"
)

print(f"Reloaded vector store from: {persist_directory}")
print(f"Collection size: {reloaded_vectorstore._collection.count()} documents")

# Verify it works
test_results = reloaded_vectorstore.similarity_search("system deployment", k=1)
print(f"\nTest query result preview: {test_results[0].page_content[:150]}...")
print("\nVector store persisted and reloaded successfully!")

Reloaded vector store from: d:\AI\agentic-ai-system\training\module2_rag\chroma_db
Collection size: 20 documents

Test query result preview: - Redis Cache (Port 6379): Provides caching for frequently used data and serves as a message queue for asynchronous task processing.

Built-in Tools
-...

Vector store persisted and reloaded successfully!


---

## Section 6: Retrieval Chain (RAG)

Now we connect all the pieces into a **RAG chain**. The chain works as follows:

```
User Question
    |                           |
    v                           v
 Retriever                  Passthrough
 (find relevant docs)       (keep question as-is)
    |                           |
    v                           v
 format_docs              question string
 (join chunks)                  |
    |                           |
    +--------> Prompt <---------+
               (combine context + question)
                  |
                  v
               LLM (Ollama)
                  |
                  v
            StrOutputParser
                  |
                  v
            Answer (string)
```

This uses LangChain Expression Language (LCEL) to compose the chain declaratively.

In [16]:
# -----------------------------------------------------------------
# Build the RAG chain
# -----------------------------------------------------------------
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# 1. Create the LLM
llm = ChatOllama(
    model="llama3.2:3b",
    base_url="http://localhost:11434",
    temperature=0.3,  # Lower temperature for more factual answers
)

# 2. Create a retriever from the vector store
# k=4 means retrieve the top 4 most relevant chunks
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}
)

# 3. Define a function to format retrieved documents into a single string
def format_docs(docs):
    """Join retrieved document chunks into a single context string."""
    return "\n\n".join(doc.page_content for doc in docs)

# 4. Create the prompt template
prompt = ChatPromptTemplate.from_template(
    """You are a helpful assistant for Alwajeez Tech. Answer the question based only 
on the provided context. If the context does not contain enough information to 
answer the question, say "I don't have enough information to answer that."

Context:
{context}

Question: {question}

Answer:"""
)

# 5. Compose the RAG chain using LCEL (LangChain Expression Language)
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("RAG chain built successfully!")
print("Components: Retriever -> format_docs -> Prompt -> LLM -> StrOutputParser")

RAG chain built successfully!
Components: Retriever -> format_docs -> Prompt -> LLM -> StrOutputParser


In [17]:
# -----------------------------------------------------------------
# Test the RAG chain with multiple questions
# -----------------------------------------------------------------
test_questions = [
    "What is ZATCA?",
    "What is the VAT rate in Saudi Arabia?",
    "How does the ReAct agent work?",
    "What tools are available in the system?",
    "How is the system deployed?",
]

for question in test_questions:
    print(f"\n{'=' * 70}")
    print(f"Q: {question}")
    print(f"{'=' * 70}")
    
    answer = rag_chain.invoke(question)
    print(f"A: {answer}")


Q: What is ZATCA?
A: ZATCA stands for Zakat, Tax and Customs Authority, the Saudi Arabian government body responsible for tax collection and customs regulation.

Q: What is the VAT rate in Saudi Arabia?
A: The standard VAT (Value Added Tax) rate in Saudi Arabia is 15%. This rate was increased from 5% to 15% in July 2020. Certain supplies are zero-rated (0%) such as exports and international transportation, while some supplies are exempt from VAT entirely, such as financial services and residential real estate.

Q: How does the ReAct agent work?
A: The ReAct agent follows a loop of Thought -> Action -> Observation. 

Here's a breakdown:

1. **Thought**: The agent thinks about the task and decides what to do.
2. **Action**: It takes an action by calling a tool (calculator, web search, file operations, etc.).
3. **Observation**: It observes the result, then decides whether to take another action or provide a final answer.

The maximum number of iterations is configurable (default: 10).



---

## Section 7: RAG FastAPI Endpoint

In production, you want to expose the RAG pipeline as an API. This section creates a standalone FastAPI server with two endpoints:

- **`POST /rag/ingest`** -- Accepts a file path, loads the document, chunks it, and stores the embeddings in ChromaDB.
- **`POST /rag/query`** -- Accepts a question string and returns the RAG-generated answer.

This server runs independently from the main Agentic AI server (on a different port).

In [18]:
# -----------------------------------------------------------------
# Write the RAG server to a file
# -----------------------------------------------------------------
import os

rag_server_code = '''"""RAG Server -- Standalone FastAPI endpoint for RAG queries.

Run with: uvicorn rag_server:app --port 8001 --reload
"""

import os
import shutil
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_chroma import Chroma
from langchain_community.document_loaders import TextLoader, PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


# --- Configuration ---
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")
OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "llama3.2:3b")
CHROMA_DIR = os.getenv("CHROMA_DIR", "./chroma_db")
COLLECTION_NAME = "alwajeez_docs"

# --- Initialize components ---
embeddings = OllamaEmbeddings(model=OLLAMA_MODEL, base_url=OLLAMA_BASE_URL)
llm = ChatOllama(model=OLLAMA_MODEL, base_url=OLLAMA_BASE_URL, temperature=0.3)

# Load or create vector store
vectorstore = Chroma(
    persist_directory=CHROMA_DIR,
    embedding_function=embeddings,
    collection_name=COLLECTION_NAME,
)

# --- Pydantic models ---
class IngestRequest(BaseModel):
    file_path: str
    chunk_size: int = 500
    chunk_overlap: int = 100


class IngestResponse(BaseModel):
    status: str
    file_path: str
    chunks_added: int
    total_documents: int


class QueryRequest(BaseModel):
    question: str
    k: int = 4


class QueryResponse(BaseModel):
    question: str
    answer: str
    sources: list[dict]


# --- Helper functions ---
def format_docs(docs):
    return "\\n\\n".join(doc.page_content for doc in docs)


# --- FastAPI app ---
app = FastAPI(
    title="RAG Server",
    description="Retrieval-Augmented Generation API for Alwajeez Tech",
    version="1.0.0",
)


@app.get("/health")
async def health():
    return {
        "status": "healthy",
        "model": OLLAMA_MODEL,
        "documents": vectorstore._collection.count(),
    }


@app.post("/rag/ingest", response_model=IngestResponse)
async def ingest(request: IngestRequest):
    """Ingest a document: load, chunk, embed, and store in ChromaDB."""
    if not os.path.exists(request.file_path):
        raise HTTPException(status_code=404, detail=f"File not found: {request.file_path}")

    # Choose loader based on file extension
    if request.file_path.endswith(".pdf"):
        loader = PyPDFLoader(request.file_path)
    else:
        loader = TextLoader(request.file_path, encoding="utf-8")

    documents = loader.load()

    # Split into chunks
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=request.chunk_size,
        chunk_overlap=request.chunk_overlap,
    )
    chunks = splitter.split_documents(documents)

    # Add to vector store
    vectorstore.add_documents(chunks)

    return IngestResponse(
        status="success",
        file_path=request.file_path,
        chunks_added=len(chunks),
        total_documents=vectorstore._collection.count(),
    )


@app.post("/rag/query", response_model=QueryResponse)
async def query(request: QueryRequest):
    """Query the RAG pipeline: retrieve relevant chunks and generate answer."""
    if vectorstore._collection.count() == 0:
        raise HTTPException(
            status_code=400,
            detail="No documents in the knowledge base. Ingest documents first.",
        )

    # Build the chain
    retriever = vectorstore.as_retriever(search_kwargs={"k": request.k})

    prompt = ChatPromptTemplate.from_template(
        """You are a helpful assistant for Alwajeez Tech. Answer the question based only
on the provided context. If the context does not contain enough information,
say "I don\'t have enough information to answer that."

Context:
{context}

Question: {question}

Answer:"""
    )

    chain = (
        {"context": retriever | format_docs, "question": RunnablePassthrough()}
        | prompt
        | llm
        | StrOutputParser()
    )

    answer = chain.invoke(request.question)

    # Also retrieve source documents for transparency
    source_docs = retriever.invoke(request.question)
    sources = [
        {
            "content": doc.page_content[:200],
            "source": doc.metadata.get("source", "unknown"),
        }
        for doc in source_docs
    ]

    return QueryResponse(
        question=request.question,
        answer=answer,
        sources=sources,
    )


if __name__ == "__main__":
    import uvicorn
    uvicorn.run("rag_server:app", host="0.0.0.0", port=8001, reload=True)
'''

server_path = os.path.join(os.getcwd(), "rag_server.py")
with open(server_path, "w", encoding="utf-8") as f:
    f.write(rag_server_code)

print(f"RAG server written to: {server_path}")
print(f"\nTo run the server:")
print(f"  cd {os.getcwd()}")
print(f"  uvicorn rag_server:app --port 8001 --reload")

RAG server written to: d:\AI\agentic-ai-system\training\module2_rag\rag_server.py

To run the server:
  cd d:\AI\agentic-ai-system\training\module2_rag
  uvicorn rag_server:app --port 8001 --reload


In [19]:
# -----------------------------------------------------------------
# How to run the RAG server (instructions)
# -----------------------------------------------------------------
# In a separate terminal, run:
#
#   cd <this notebook's directory>
#   uvicorn rag_server:app --port 8001 --reload
#
# The server will be available at http://localhost:8001
# You can check health at: http://localhost:8001/health
#
# API docs (Swagger UI): http://localhost:8001/docs

print("RAG Server Endpoints:")
print("  GET  /health      -- Check server health and document count")
print("  POST /rag/ingest  -- Ingest a document into the knowledge base")
print("  POST /rag/query   -- Ask a question and get a RAG-powered answer")
print("  GET  /docs        -- Swagger UI (interactive API docs)")

RAG Server Endpoints:
  GET  /health      -- Check server health and document count
  POST /rag/ingest  -- Ingest a document into the knowledge base
  POST /rag/query   -- Ask a question and get a RAG-powered answer
  GET  /docs        -- Swagger UI (interactive API docs)


In [20]:
# -----------------------------------------------------------------
# Test the RAG server endpoints with httpx
# -----------------------------------------------------------------
# Note: This cell will only work if the rag_server is running on port 8001.
# If the server is not running, the requests will fail gracefully.

import httpx
import os

BASE_URL = "http://localhost:8001"

async def test_rag_server():
    async with httpx.AsyncClient(timeout=60.0) as client:
        # Test 1: Health check
        print("--- Test 1: Health Check ---")
        try:
            resp = await client.get(f"{BASE_URL}/health")
            print(f"Status: {resp.status_code}")
            print(f"Response: {resp.json()}")
        except httpx.ConnectError:
            print("Server not running. Start it with: uvicorn rag_server:app --port 8001")
            return

        # Test 2: Ingest a document
        print("\n--- Test 2: Ingest Document ---")
        docs_dir = os.path.join(os.getcwd(), "sample_documents")
        ingest_payload = {
            "file_path": os.path.join(docs_dir, "company_overview.txt"),
            "chunk_size": 500,
            "chunk_overlap": 100,
        }
        resp = await client.post(f"{BASE_URL}/rag/ingest", json=ingest_payload)
        print(f"Status: {resp.status_code}")
        print(f"Response: {resp.json()}")

        # Test 3: Query the RAG pipeline
        print("\n--- Test 3: RAG Query ---")
        query_payload = {
            "question": "What is ZATCA and what are its phases?",
            "k": 3,
        }
        resp = await client.post(f"{BASE_URL}/rag/query", json=query_payload)
        print(f"Status: {resp.status_code}")
        result = resp.json()
        print(f"Question: {result['question']}")
        print(f"Answer: {result['answer']}")
        print(f"Sources: {len(result['sources'])} chunks used")


# Run the async test
import asyncio

# Use nest_asyncio for Jupyter compatibility
try:
    import nest_asyncio
    nest_asyncio.apply()
except ImportError:
    pass  # nest_asyncio not required if not in a running event loop

asyncio.run(test_rag_server())

--- Test 1: Health Check ---
Status: 200
Response: {'status': 'healthy', 'model': 'llama3.2:3b', 'documents': 20}

--- Test 2: Ingest Document ---
Status: 200
Response: {'status': 'success', 'file_path': 'd:\\AI\\agentic-ai-system\\training\\module2_rag\\sample_documents\\company_overview.txt', 'chunks_added': 8, 'total_documents': 28}

--- Test 3: RAG Query ---
Status: 200
Question: What is ZATCA and what are its phases?
Answer: I don't have enough information to answer that.
Sources: 3 chunks used


---

## Section 8: Conversational RAG

The basic RAG chain treats each question independently. But in real conversations, users ask **follow-up questions** that reference earlier context:

- User: *"What is ZATCA?"*
- Assistant: *"ZATCA stands for Zakat, Tax and Customs Authority..."*
- User: *"What are its two phases?"*  <-- "its" refers to ZATCA

To handle this, we need **two stages**:

1. **Contextualize the question:** Given the chat history, reformulate the latest question into a standalone question. For example, *"What are its two phases?"* becomes *"What are the two phases of ZATCA e-invoicing?"*

2. **Answer the standalone question:** Use the normal RAG chain with the reformulated question.

This approach is sometimes called **History-Aware Retrieval**.

In [21]:
# -----------------------------------------------------------------
# Build Conversational RAG chain
# -----------------------------------------------------------------
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough, RunnableLambda
from langchain_core.messages import HumanMessage, AIMessage

# LLM instance
llm = ChatOllama(
    model="llama3.2:3b",
    base_url="http://localhost:11434",
    temperature=0.3,
)

# --- Stage 1: Contextualize the question ---
# This prompt takes chat history + latest question and produces a standalone question.
contextualize_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "Given the following chat history and the latest user question, "
     "reformulate the question into a standalone question that can be "
     "understood without the chat history. Do NOT answer the question -- "
     "only reformulate it if needed. If it is already standalone, return it as-is."),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

# Chain that contextualizes the question
contextualize_chain = contextualize_prompt | llm | StrOutputParser()

# --- Stage 2: RAG answer ---
# This prompt takes context (retrieved docs) + chat history + question and produces an answer.
qa_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a helpful assistant for Alwajeez Tech. Answer the question based only "
     "on the provided context. If the context does not contain enough information, "
     "say 'I don't have enough information to answer that.'\n\n"
     "Context:\n{context}"),
    MessagesPlaceholder("chat_history"),
    ("human", "{input}"),
])

# Retriever
retriever = vectorstore.as_retriever(search_kwargs={"k": 4})


def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


def conversational_rag(input_dict):
    """Run the full conversational RAG pipeline."""
    chat_history = input_dict.get("chat_history", [])
    user_input = input_dict["input"]

    # Stage 1: Contextualize if there is chat history
    if chat_history:
        standalone_question = contextualize_chain.invoke({
            "chat_history": chat_history,
            "input": user_input,
        })
        print(f"  [Reformulated question: '{standalone_question}']")
    else:
        standalone_question = user_input

    # Stage 2: Retrieve and answer
    retrieved_docs = retriever.invoke(standalone_question)
    context = format_docs(retrieved_docs)

    answer = (qa_prompt | llm | StrOutputParser()).invoke({
        "context": context,
        "chat_history": chat_history,
        "input": user_input,
    })

    return answer


print("Conversational RAG chain built successfully!")
print("It will reformulate follow-up questions using chat history.")

Conversational RAG chain built successfully!
It will reformulate follow-up questions using chat history.


In [22]:
# -----------------------------------------------------------------
# Test multi-turn conversation
# -----------------------------------------------------------------
from langchain_core.messages import HumanMessage, AIMessage

# Maintain chat history as a list of messages
chat_history = []

# Define a sequence of questions that test context-awareness
conversation = [
    "What is ZATCA?",
    "What are its two phases?",       # "its" should refer to ZATCA
    "What is the VAT rate?",           # New topic
    "When was it changed?",            # "it" should refer to the VAT rate
]

for i, question in enumerate(conversation):
    print(f"\n{'=' * 70}")
    print(f"Turn {i + 1} -- User: {question}")
    print(f"{'=' * 70}")

    # Run the conversational RAG chain
    answer = conversational_rag({
        "input": question,
        "chat_history": chat_history,
    })

    print(f"\nAssistant: {answer}")

    # Update chat history
    chat_history.append(HumanMessage(content=question))
    chat_history.append(AIMessage(content=answer))

print(f"\n{'=' * 70}")
print(f"Conversation complete. Total turns: {len(conversation)}")
print(f"Chat history length: {len(chat_history)} messages")


Turn 1 -- User: What is ZATCA?

Assistant: ZATCA stands for Zakat, Tax and Customs Authority, which is the Saudi Arabian government body responsible for tax collection and customs regulation.

Turn 2 -- User: What are its two phases?
  [Reformulated question: 'I couldn't find any information on the two phases of ZATCA. Can you please provide more context or clarify what you mean by "phases"? I'll do my best to help. 

(Revised reformulation) What are the phases of Zakat, Tax and Customs Authority (ZATCA)?']

Assistant: I don't have enough information to answer that based on the provided context. The only phase mentioned in the context is Phase 1, which requires generating invoices in a specific format.

Turn 3 -- User: What is the VAT rate?
  [Reformulated question: 'The VAT (Value-Added Tax) rate in Saudi Arabia, administered by ZATCA, is 5%.']

Assistant: The standard VAT (Value Added Tax) rate in Saudi Arabia is 15%. This rate was increased from 5% to 15% in July 2020. Certain supp

---

## Section 9: RAG Evaluation

### Why Evaluate RAG?

Building a RAG pipeline is only the first step. You need to **measure its quality** to:
- Compare different chunking strategies
- Test different embedding models
- Ensure the system answers accurately for your domain
- Detect regressions when you change the pipeline

### Simple Keyword-Based Evaluation

For this lab, we use a lightweight approach: define expected keywords for each question, then check how many keywords appear in the RAG answer. This is not a perfect metric (LLM-based evaluation is better), but it is fast and gives a useful signal.

**Scoring formula:**
```
score = (number of expected keywords found in answer) / (total expected keywords)
```

In [24]:
# -----------------------------------------------------------------
# Define evaluation set with expected keywords
# -----------------------------------------------------------------
eval_set = [
    {
        "question": "What is the VAT rate in Saudi Arabia?",
        "expected_keywords": ["15%", "VAT"],
    },
    {
        "question": "What port does the API run on?",
        "expected_keywords": ["8000", "FastAPI"],
    },
    {
        "question": "What is ZATCA?",
        "expected_keywords": ["Zakat", "Tax", "Customs"],
    },
    {
        "question": "How does the ReAct agent work?",
        "expected_keywords": ["Thought", "Action", "Observation"],
    },
    {
        "question": "What tools does the system provide?",
        "expected_keywords": ["Calculator", "Web Search", "File"],
    },
    {
        "question": "How is the system deployed?",
        "expected_keywords": ["Docker", "Compose"],
    },
    {
        "question": "What is Alwajeez Tech?",
        "expected_keywords": ["Saudi Arabia", "AI"],
    },
    {
        "question": "What model does the system use by default?",
        "expected_keywords": ["llama3.2", "3b"],
    },
]

print(f"Evaluation set: {len(eval_set)} questions")
for item in eval_set:
    print(f"  Q: {item['question']}")
    print(f"     Expected: {item['expected_keywords']}")

Evaluation set: 8 questions
  Q: What is the VAT rate in Saudi Arabia?
     Expected: ['15%', 'VAT']
  Q: What port does the API run on?
     Expected: ['8000', 'FastAPI']
  Q: What is ZATCA?
     Expected: ['Zakat', 'Tax', 'Customs']
  Q: How does the ReAct agent work?
     Expected: ['Thought', 'Action', 'Observation']
  Q: What tools does the system provide?
     Expected: ['Calculator', 'Web Search', 'File']
  Q: How is the system deployed?
     Expected: ['Docker', 'Compose']
  Q: What is Alwajeez Tech?
     Expected: ['Saudi Arabia', 'AI']
  Q: What model does the system use by default?
     Expected: ['llama3.2', '3b']


In [25]:
# -----------------------------------------------------------------
# Run evaluation across all chunking configurations
# -----------------------------------------------------------------
import os
import shutil
from langchain_chroma import Chroma
from langchain_ollama import OllamaEmbeddings, ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough


def score_answer(answer, expected_keywords):
    """Score an answer based on the presence of expected keywords (case-insensitive)."""
    answer_lower = answer.lower()
    found = sum(1 for kw in expected_keywords if kw.lower() in answer_lower)
    return found / len(expected_keywords) if expected_keywords else 0.0


def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


# Embeddings and LLM
eval_embeddings = OllamaEmbeddings(
    model="llama3.2:3b",
    base_url="http://localhost:11434"
)
eval_llm = ChatOllama(
    model="llama3.2:3b",
    base_url="http://localhost:11434",
    temperature=0.1,  # Very low temperature for consistent evaluation
)

eval_prompt = ChatPromptTemplate.from_template(
    """Answer the question based only on the provided context. Be specific and include 
relevant details like numbers, names, and technical terms.

Context:
{context}

Question: {question}

Answer:"""
)

# Store results for each config
eval_results = {}

for config_name, chunks in chunking_results.items():
    print(f"\nEvaluating {config_name}...")

    # Create a temporary vector store for this config
    eval_persist_dir = os.path.join(os.getcwd(), f"chroma_eval_{config_name.replace(' ', '_')}")
    if os.path.exists(eval_persist_dir):
        shutil.rmtree(eval_persist_dir)

    eval_vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=eval_embeddings,
        persist_directory=eval_persist_dir,
    )

    eval_retriever = eval_vectorstore.as_retriever(search_kwargs={"k": 4})

    eval_chain = (
        {"context": eval_retriever | format_docs, "question": RunnablePassthrough()}
        | eval_prompt
        | eval_llm
        | StrOutputParser()
    )

    config_scores = []
    for item in eval_set:
        answer = eval_chain.invoke(item["question"])
        s = score_answer(answer, item["expected_keywords"])
        config_scores.append(s)
        print(f"  Q: {item['question'][:50]:50s} Score: {s:.2f}")

    avg_score = sum(config_scores) / len(config_scores) if config_scores else 0.0
    eval_results[config_name] = {
        "scores": config_scores,
        "avg_score": avg_score,
        "num_chunks": len(chunks),
    }
    print(f"  Average score: {avg_score:.2f}")

    # Clean up temporary vector store
    if os.path.exists(eval_persist_dir):
        shutil.rmtree(eval_persist_dir)


Evaluating Config A (100/20)...
  Q: What is the VAT rate in Saudi Arabia?              Score: 1.00
  Q: What port does the API run on?                     Score: 0.00
  Q: What is ZATCA?                                     Score: 0.00
  Q: How does the ReAct agent work?                     Score: 0.33
  Q: What tools does the system provide?                Score: 0.00
  Q: How is the system deployed?                        Score: 0.00
  Q: What is Alwajeez Tech?                             Score: 0.50
  Q: What model does the system use by default?         Score: 0.00
  Average score: 0.23


PermissionError: [WinError 32] The process cannot access the file because it is being used by another process: 'd:\\AI\\agentic-ai-system\\training\\module2_rag\\chroma_eval_Config_A_(100/20)\\eae35e4e-3e96-49db-b0ce-d6833a07fbfd\\data_level0.bin'

In [ ]:
# -----------------------------------------------------------------
# Print comparison table
# -----------------------------------------------------------------
print(f"\n{'=' * 65}")
print(f"RAG EVALUATION RESULTS")
print(f"{'=' * 65}")
print(f"{'Config':<25} {'Chunks':>8} {'Avg Score':>12} {'Accuracy':>10}")
print(f"{'-' * 65}")

best_config = None
best_score = -1

for config_name, result in eval_results.items():
    accuracy_pct = result['avg_score'] * 100
    marker = ""
    if result['avg_score'] > best_score:
        best_score = result['avg_score']
        best_config = config_name

    print(
        f"{config_name:<25} {result['num_chunks']:>8} "
        f"{result['avg_score']:>12.3f} {accuracy_pct:>9.1f}%"
    )

print(f"{'-' * 65}")
print(f"\nBest configuration: {best_config} (score: {best_score:.3f})")

# Per-question breakdown for best config
print(f"\n{'=' * 65}")
print(f"Per-Question Breakdown for {best_config}")
print(f"{'=' * 65}")
best_scores = eval_results[best_config]['scores']
for i, item in enumerate(eval_set):
    status = "PASS" if best_scores[i] >= 0.5 else "FAIL"
    print(f"  [{status}] {item['question'][:50]:50s} {best_scores[i]:.2f}")


RAG EVALUATION RESULTS
Config                      Chunks    Avg Score   Accuracy
-----------------------------------------------------------------
Config A (100/20)              110        0.229      22.9%
-----------------------------------------------------------------

Best configuration: Config A (100/20) (score: 0.229)

Per-Question Breakdown for Config A (100/20)
  [PASS] What is the VAT rate in Saudi Arabia?              1.00
  [FAIL] What port does the API run on?                     0.00
  [FAIL] What is ZATCA?                                     0.00
  [FAIL] How does the ReAct agent work?                     0.33
  [FAIL] What tools does the system provide?                0.00
  [FAIL] How is the system deployed?                        0.00
  [PASS] What is Alwajeez Tech?                             0.50
  [FAIL] What model does the system use by default?         0.00


: 

---

## Summary & Best Practices

### What We Built

In this lab, we built a complete RAG pipeline from scratch:

1. **Document Loading** -- Used LangChain loaders (TextLoader, DirectoryLoader) to read files.
2. **Chunking** -- Split documents using RecursiveCharacterTextSplitter with different configs.
3. **Embeddings** -- Converted text chunks into vectors using Ollama (llama3.2:3b).
4. **Vector Storage** -- Stored and queried embeddings with ChromaDB.
5. **RAG Chain** -- Combined retriever + prompt + LLM into a working Q&A pipeline.
6. **API Server** -- Packaged the RAG pipeline as a FastAPI service.
7. **Conversational RAG** -- Added chat history awareness for follow-up questions.
8. **Evaluation** -- Compared chunking strategies using keyword-based scoring.

### Best Practices

| Area | Recommendation |
|------|---------------|
| **Chunking** | Start with 500/100 and adjust based on evaluation. Smaller chunks for precise Q&A, larger for summaries. |
| **Retrieval** | Use `k=3-5` for most use cases. Too many chunks dilute relevance; too few may miss information. |
| **Prompt** | Always instruct the LLM to answer based only on the provided context. This reduces hallucination. |
| **Evaluation** | Build an evaluation set early and run it after every pipeline change. |
| **Embeddings** | The embedding model matters more than the LLM for retrieval quality. Consider dedicated embedding models. |
| **Production** | Use persistent ChromaDB storage. Add metadata filtering. Monitor retrieval quality over time. |

### Next Steps

- **Module 3:** Fine-tuning Ollama models for domain-specific tasks
- **Module 4:** RLHF and DPO for improving model alignment
- Try integrating this RAG pipeline into the main Agentic AI System as a new tool
- Experiment with different embedding models and chunking strategies